## Build our first baseline
We implement a question-only baseline first here. It will use the question text without the image, providing a reference point for how much of VizWiz can be predicted using language patterns alone.

## Question-only VQA baseline (VizWiz)

**Goal:** quantify how well questions alone predict answers, before using images with BLIP. This is a language-bias baseline, not the main multimodal model.

### Setup, paths, reproducibility

In [1]:
import json, random, re
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix
import joblib

SEED = 740
random.seed(SEED)
np.random.seed(SEED)

# Works when notebook is launched from Project/ or Project/notebooks/
cwd = Path.cwd()
root = next((p for p in [cwd, cwd.parent] if (p / 'data' / 'Annotations' / 'train.json').exists()), None)
if root is None:
    raise FileNotFoundError(f'Could not locate training annotations from {cwd}')
DATA_DIR = root / 'data' / 'Annotations'
RESULTS_DIR = root / 'results' / 'question_only'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('Project root:', root)
print('Train:', DATA_DIR / 'train.json')

Project root: c:\Users\GugurGubil\Bita Drive\1. McMaster\1. Hamid Deep Learning\Project
Train: c:\Users\GugurGubil\Bita Drive\1. McMaster\1. Hamid Deep Learning\Project\data\Annotations\train.json


### Read and audit train/validation files

In [3]:
def load_split(name):
    with (DATA_DIR / f'{name}.json').open(encoding='utf-8') as f:
        return pd.DataFrame(json.load(f))

train_df = load_split('train')
val_df = load_split('val')

print('Train:', train_df.shape, 'Validation:', val_df.shape)
print('Train answerability:', train_df['answerable'].value_counts().to_dict())
print('Validation answerability:', val_df['answerable'].value_counts().to_dict())
print('Overlapping image filenames:', len(set(train_df['image']) & set(val_df['image'])))


Train: (20523, 5) Validation: (4319, 5)
Train answerability: {1: 14991, 0: 5532}
Validation answerability: {1: 2934, 0: 1385}
Overlapping image filenames: 0


### Target preparation and label audits

For this *simple* baseline, we use the most frequent exact normalized human answer as a single label. In ties, `Counter.most_common` follows first appearance. Preserve all ten references for scoring. We do **not** alter the target according to the `answerable` field; the 2,056 observed conflicts remain an analysis concern.

In [4]:
def normalize_simple(text):
    return re.sub(r'\s+', ' ', str(text).strip().lower())

def reference_answers(row):
    return [normalize_simple(a['answer']) for a in row['answers']]

def majority_answer(row):
    return Counter(reference_answers(row)).most_common(1)[0][0]

for frame in [train_df, val_df]:
    frame['target'] = frame.apply(majority_answer, axis=1)
    frame['question_text'] = frame['question'].astype(str).str.strip()

print('Top train targets:', train_df['target'].value_counts().head(10).to_dict())
print('Train: majority unanswerable despite answerable=1:',
      ((train_df['answerable'] == 1) & (train_df['target'] == 'unanswerable')).sum())


Top train targets: {'unanswerable': 7588, 'no': 508, 'yes': 501, 'white': 294, 'grey': 286, 'black': 228, 'blue': 214, 'red': 129, 'brown': 101, 'pink': 99}
Train: majority unanswerable despite answerable=1: 2056


### VQA-style evaluation

The standard leave-one-out agreement score averages each reference annotator’s agreement with the other nine: `min(matches/3, 1)`. This function does **not** implement the official VQA punctuation, article and numeric normalization; use one fixed evaluator for comparisons.

In [5]:
def vqa_style_score(prediction, answers):
    p = normalize_simple(prediction)
    refs = [normalize_simple(a['answer']) for a in answers]
    return float(np.mean([
        min(sum(p == ref for j, ref in enumerate(refs) if j != i) / 3.0, 1.0)
        for i in range(len(refs))
    ]))

def evaluate(predictions, frame):
    scores = np.array([
        vqa_style_score(p, answers)
        for p, answers in zip(predictions, frame['answers'])
    ])
    answerable = frame['answerable'].to_numpy(dtype=bool)
    return {
        'overall_vqa_style': float(scores.mean()),
        'answerable_vqa_style': float(scores[answerable].mean()),
        'unanswerable_vqa_style': float(scores[~answerable].mean()),
        'exact_majority_accuracy': float(np.mean(
            np.asarray(predictions) == frame['target'].to_numpy()
        )),
    }, scores


### Constant-answer baseline (no training)

In [6]:
majority_train = train_df['target'].value_counts().idxmax()
constant_predictions = [majority_train] * len(val_df)
constant_metrics, constant_scores = evaluate(constant_predictions, val_df)
print('Constant label:', majority_train)
print(pd.Series(constant_metrics).round(4))

Constant label: unanswerable
overall_vqa_style          0.5212
answerable_vqa_style       0.2952
unanswerable_vqa_style     1.0000
exact_majority_accuracy    0.4332
dtype: float64


### TF-IDF + linear classifier

**Top-K vocabulary is derived from the training split only.** For this simple classifier, rows whose majority answer is outside that vocabulary are omitted *from classifier fitting only*. The model is still evaluated on **every** validation question, so restricted coverage remains visible. We report coverage separately for all and answerable training questions.

In [7]:
K = 1000
counts = train_df['target'].value_counts()
vocab = set(counts.head(K).index)
fit_df = train_df[train_df['target'].isin(vocab)].copy()

print('Training rows retained:', len(fit_df), '/', len(train_df))
print('All-train target coverage:', round(train_df['target'].isin(vocab).mean(), 4))
print('Answerable-train target coverage:', round(
    train_df.loc[train_df['answerable'] == 1, 'target'].isin(vocab).mean(), 4
))
print('Label classes fitted:', fit_df['target'].nunique())


Training rows retained: 15781 / 20523
All-train target coverage: 0.7689
Answerable-train target coverage: 0.6837
Label classes fitted: 1000


In [8]:
model = Pipeline([
    ('tfidf', TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2,
                               max_features=20000, sublinear_tf=True)),
    ('classifier', LinearSVC(C=1.0, random_state=SEED, max_iter=5000))
])
model.fit(fit_df['question_text'], fit_df['target'])
predictions = model.predict(val_df['question_text'])
model_metrics, model_scores = evaluate(predictions, val_df)
print(pd.DataFrame([constant_metrics, model_metrics], index=['Constant', 'TF-IDF + LinearSVC']).round(4))

                    overall_vqa_style  answerable_vqa_style  \
Constant                       0.5212                0.2952   
TF-IDF + LinearSVC             0.5255                0.3213   

                    unanswerable_vqa_style  exact_majority_accuracy  
Constant                            1.0000                   0.4332  
TF-IDF + LinearSVC                  0.9583                   0.4427  


### Analyze error patterns

This baseline deliberately receives **only question text**; it cannot inspect images. Review its predictions for common language priors and answerability failures.


In [9]:
error_df = val_df[['image', 'question', 'answerable', 'target']].copy()
error_df['prediction'] = predictions
error_df['vqa_style_score'] = model_scores
error_df['top_answer_votes'] = val_df['answers'].apply(
    lambda ans: Counter(normalize_simple(a['answer']) for a in ans).most_common(1)[0][1]
)
print('Most common predicted answers:', Counter(predictions).most_common(12))
print('Ten low-scoring examples:')
display(error_df.sort_values(['vqa_style_score', 'top_answer_votes']).head(10))


Most common predicted answers: [('unanswerable', 3701), ('grey', 116), ('no', 99), ('white', 76), ('yes', 69), ('black', 32), ('blue', 19), ('cloudy', 12), ('red', 10), ('20', 8), ('brown', 8), ('dog', 6)]
Ten low-scoring examples:


,image,question,answerable,target,prediction,vqa_style_score,top_answer_votes
21,VizWiz_val_00000021.jpg,What is this color and pattern? There's some s...,1,plain white uspa logo in gold,unanswerable,0.0,1
112,VizWiz_val_00000112.jpg,Please describe this page. Thank you.,1,store ad,unanswerable,0.0,1
137,VizWiz_val_00000137.jpg,What is on this piece of paper? Thank you.,1,sales ad,unanswerable,0.0,1
140,VizWiz_val_00000140.jpg,Can you please describe to me what's on the pa...,1,sideways car sales ad,unanswerable,0.0,1
272,VizWiz_val_00000272.jpg,Can you please tell me what's on the screen? A...,1,windows 7 homechoose country time currency key...,unanswerable,0.0,1
370,VizWiz_val_00000370.jpg,Could you tell me if I need to hit enter or ca...,1,yes accept terms,unanswerable,0.0,1
549,VizWiz_val_00000549.jpg,Can you describe what this page looks like? Th...,1,white 2 lines text,wreath,0.0,1
588,VizWiz_val_00000588.jpg,Is this the google satellite image and I would...,1,overlooking neighborhood park in,unanswerable,0.0,1
692,VizWiz_val_00000692.jpg,What can you see?,1,person that barely in image head arm,sink,0.0,1
708,VizWiz_val_00000708.jpg,What is this?,1,coupons,unanswerable,0.0,1


### Save results and trained baseline

Store small derived results and model artifacts outside `data/`; avoid committing large binary models to Git. Change K or hyperparameters only after recording the initial baseline; validation remains for model selection, never the test answers.

In [10]:
metrics_df = pd.DataFrame([constant_metrics, model_metrics], index=['constant', 'tfidf_linearsvc'])
metrics_df.to_csv(RESULTS_DIR / 'validation_metrics.csv')
error_df.to_csv(RESULTS_DIR / 'validation_predictions.csv', index=False)
joblib.dump({'model': model, 'seed': SEED, 'k': K}, RESULTS_DIR / 'tfidf_linearsvc.joblib')
print('Saved to:', RESULTS_DIR)

Saved to: c:\Users\GugurGubil\Bita Drive\1. McMaster\1. Hamid Deep Learning\Project\results\question_only


In [11]:
answerable_mask = val_df["answerable"] == 1

unanswerable_on_answerable = (
    predictions[answerable_mask.to_numpy()] == "unanswerable"
).mean()

print(
    "Unanswerable predictions on answerable questions:",
    f"{unanswerable_on_answerable:.2%}"
)

Unanswerable predictions on answerable questions: 81.49%


## Results and Discussion

### 1. Experimental Setup

We evaluated two question-only baselines on the VizWiz dataset to establish how well answers can be predicted without visual information.

The training set contained 20,523 examples, and the validation set contained 4,319 examples. A constant-answer baseline was compared against a TF-IDF + LinearSVC classifier using the top 1,000 most frequent training majority-answer labels.

Both models were evaluated using VQA-style agreement scores calculated against all ten human reference answers, alongside exact majority-answer accuracy. The VQA-style evaluator uses simplified text normalization rather than the complete official VizWiz normalization procedure.

### 2. Quantitative Results

| Metric | Constant Baseline | TF-IDF + LinearSVC |
|---|---:|---:|
| Overall VQA-style accuracy | 52.12% | 52.55% |
| Answerable VQA-style accuracy | 29.52% | 32.13% |
| Unanswerable VQA-style accuracy | 100.00% | 95.83% |
| Exact majority-answer accuracy | 43.32% | 44.27% |

The TF-IDF classifier achieved a small overall improvement of 0.43 percentage points over the constant-answer baseline. Its performance on answerable questions improved by 2.61 percentage points, while performance on unanswerable questions decreased by 4.17 percentage points.

### 3. Class Imbalance and Prediction Bias

The most frequent training target was `"unanswerable"`, making it the prediction of the constant baseline.

The TF-IDF classifier also demonstrated a strong preference for this class, predicting `"unanswerable"` for 3,701 of the 4,319 validation questions (85.69%), despite only 32.07% of validation questions being officially labeled unanswerable.

This indicates that the classifier relies heavily on the dominant training label. The high overall VQA-style score should therefore not be interpreted as evidence of strong visual question-answering capability.

### 4. Error Analysis

Inspection of low-scoring validation predictions revealed that the classifier frequently predicted `"unanswerable"` for questions about documents, screen content, objects, and visual descriptions.

These errors are consistent with the model's lack of image input. Although TF-IDF captures lexical patterns in the questions, it cannot determine which objects are present, recognize colours, read text in images, or evaluate visual answerability.

The exact-match answer-label formulation also introduces limitations because semantically similar human answers may be represented as separate classes.

### 5. Conclusions

The question-only baselines demonstrate that language patterns and answer-frequency imbalance can produce relatively high overall VQA-style scores without using images.

However, the modest improvement from TF-IDF classification and its heavy reliance on `"unanswerable"` highlight the limitations of text-only prediction.

These results establish a reference point for evaluating whether a multimodal VQA model benefits from visual information. The next experiment will evaluate pretrained BLIP on the same validation split, followed by fine-tuning and comparison of its performance on answerable and unanswerable questions.

All final model comparisons will use a consistent evaluation procedure, with official VizWiz answer normalization incorporated before final reporting.

### Additional Analysis: Unanswerable Prediction Bias

The TF-IDF + LinearSVC classifier predicted `"unanswerable"` for **81.49% of officially answerable validation questions**. This indicates a strong tendency to predict the dominant training answer rather than producing specific responses to visual questions.

The result suggests that question text alone provides insufficient information for many VizWiz examples. Without image features, the classifier cannot reliably identify visible objects, read text, recognize colours, or determine whether the requested visual information is available.

Although predictions of `"unanswerable"` may receive partial VQA credit when some annotators provide that response, the high prediction rate demonstrates a substantial limitation of the question-only baseline.

This finding motivates evaluating a multimodal model such as BLIP, which incorporates both image and question information. Comparing pretrained and fine-tuned BLIP against this baseline will help quantify the benefit of visual information and domain adaptation.